# 01 - Obtain: Pengambilan dan Dokumentasi Data

**Dikerjakan oleh:** Nicholas Zefanya Lamtyo N (18223111) - Data Engineer  
**Kelompok 14 | II4013 Data Analitik**

## 1. Pertanyaan Analitik Utama

1. Bagaimana distribusi tiket layanan TI berdasarkan prioritas, tipe, dan severity?
2. Faktor apa saja yang paling memengaruhi lama penyelesaian tiket?
3. Bagaimana tren volume tiket dan kepuasan pengguna dari waktu ke waktu?
4. Kategori tiket mana yang paling sering mengalami eskalasi atau re-open?
5. Apa akar masalah utama yang menyebabkan SLA tidak terpenuhi?

## 2. Dokumentasi Dataset

| No | Nama Dataset | Sumber | URL | Periode | Format | Baris | Kolom | Level | Peran | Alasan Pemilihan |
|----|-------------|--------|-----|---------|--------|-------|-------|-------|-------|------------------|
| 1 | WA_Fn-UseC_-IT-Help-Desk.csv | Tiket helpdesk TI internal yang telah dianonimkan (Kaggle) | https://www.kaggle.com/datasets/sudhanshu746/analyze-helpdesk-tickets | Tidak diketahui | CSV | 100.000 | 10 | Organisasi | Utama | Memiliki kolom severity, priority, daysOpen, dan satisfaction yang langsung relevan untuk analisis kinerja tiket IT |
| 2 | issues.csv | Real-world helpdesk system (Kaggle) | https://www.kaggle.com/datasets/esmaeil391/it-helpdesk-tickets | Jan 2016 - Mar 2023 | CSV | 66.691 | 57 | Organisasi | Pendukung | Data nyata dengan dimensi waktu lengkap, memungkinkan analisis SLA, workflow, dan tren tiket dari waktu ke waktu |

In [13]:
import pandas as pd
import numpy as np
import os

PRIMARY_PATH    = '../data/raw/primary/WA_Fn-UseC_-IT-Help-Desk.csv'
SUPPORTING_PATH = '../data/raw/supporting/issues.csv'

## 3. Load Dataset

In [14]:
df_primary    = pd.read_csv(PRIMARY_PATH)
df_supporting = pd.read_csv(SUPPORTING_PATH)

print('Dataset utama     :', df_primary.shape)
print('Dataset pendukung :', df_supporting.shape)

Dataset utama     : (100000, 10)
Dataset pendukung : (66691, 58)


## 4. Inspeksi Dataset Utama

In [15]:
df_primary.head()

,ticket,requestor,RequestorSeniority,ITOwner,FiledAgainst,TicketType,Severity,Priority,daysOpen,Satisfaction
0,1,1929,1 - Junior,50,Systems,Issue,2 - Normal,0 - Unassigned,3,1 - Unsatisfied
1,2,1587,2 - Regular,15,Software,Request,1 - Minor,1 - Low,5,1 - Unsatisfied
2,3,925,2 - Regular,15,Access/Login,Request,2 - Normal,0 - Unassigned,0,0 - Unknown
3,4,413,4 - Management,22,Systems,Request,2 - Normal,0 - Unassigned,20,0 - Unknown
4,5,318,1 - Junior,22,Access/Login,Request,2 - Normal,1 - Low,1,1 - Unsatisfied


In [16]:
df_primary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   ticket              100000 non-null  int64 
 1   requestor           100000 non-null  int64 
 2   RequestorSeniority  100000 non-null  object
 3   ITOwner             100000 non-null  int64 
 4   FiledAgainst        100000 non-null  object
 5   TicketType          100000 non-null  object
 6   Severity            100000 non-null  object
 7   Priority            100000 non-null  object
 8   daysOpen            100000 non-null  int64 
 9   Satisfaction        100000 non-null  object
dtypes: int64(4), object(6)
memory usage: 7.6+ MB


In [17]:
df_primary.describe(include='all')

,ticket,requestor,RequestorSeniority,ITOwner,FiledAgainst,TicketType,Severity,Priority,daysOpen,Satisfaction
count,100000.000000,100000.000000,100000,100000.000000,100000,100000,100000,100000,100000.000000,100000
unique,NaN,NaN,4,NaN,4,2,5,4,NaN,4
top,NaN,NaN,2 - Regular,NaN,Systems,Request,2 - Normal,3 - High,NaN,0 - Unknown
freq,NaN,NaN,41303,NaN,40035,75074,90912,36498,NaN,30211
mean,50000.500000,999.030670,NaN,25.461000,NaN,NaN,NaN,NaN,6.842830,NaN
std,28867.657797,577.507916,NaN,14.447961,NaN,NaN,NaN,NaN,7.377876,NaN
min,1.000000,1.000000,NaN,1.000000,NaN,NaN,NaN,NaN,0.000000,NaN
25%,25000.750000,499.000000,NaN,13.000000,NaN,NaN,NaN,NaN,1.000000,NaN
50%,50000.500000,999.000000,NaN,26.000000,NaN,NaN,NaN,NaN,5.000000,NaN
75%,75000.250000,1499.000000,NaN,38.000000,NaN,NaN,NaN,NaN,10.000000,NaN


In [18]:
# Missing values
missing_primary = df_primary.isnull().sum()
missing_primary[missing_primary > 0]

Series([], dtype: int64)

In [19]:
# Jumlah nilai unik per kolom
df_primary.nunique()

ticket                100000
requestor               2000
RequestorSeniority         4
ITOwner                   50
FiledAgainst               4
TicketType                 2
Severity                   5
Priority                   4
daysOpen                  49
Satisfaction               4
dtype: int64

## 5. Inspeksi Dataset Pendukung

In [20]:
df_supporting.head()

,id,started,ended,issue_num,issue_proj,issue_reporter,issue_assignee,issue_contr_count,issue_type,issue_priority,...,wf_cancelled,wfe_cancelled,wf_under_review,wfe_under_review,wf_approved,wfe_approved,wf_pending_deployment,wfe_pending_deployment,wf_total_time,processing_steps
0,11887.0,2016-01-06 08:23:43+00:00,2016-01-06 08:56:55+00:00,186.0,d1z0,4olg,NaN,1.0,Ticket,Medium,...,NaN,0,NaN,0,NaN,0,NaN,0,1992.0,2
1,11890.0,2016-01-11 10:06:19+00:00,2016-01-12 12:30:23+00:00,190.0,d1z0,4olg,NaN,1.0,Ticket,Medium,...,NaN,0,NaN,0,NaN,0,NaN,0,95044.0,2
2,11904.0,2016-01-21 07:28:20+00:00,2016-01-26 08:21:47+00:00,198.0,d1z0,4ohk,4ohk,1.0,Ticket,Medium,...,NaN,0,NaN,0,NaN,0,NaN,0,435207.0,2
3,11907.0,2016-01-26 07:44:54+00:00,2016-01-26 07:45:48+00:00,209.0,d1z0,4olg,NaN,1.0,Vacation,Medium,...,NaN,0,NaN,0,NaN,0,NaN,0,54.0,2
4,11912.0,2016-02-01 13:45:47+00:00,2016-02-07 06:21:42+00:00,217.0,d1z0,4ohk,4ohk,1.0,Story,Medium,...,NaN,0,NaN,0,NaN,0,NaN,0,491755.0,2


In [21]:
df_supporting.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66691 entries, 0 to 66690
Data columns (total 58 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id                             66691 non-null  float64
 1   started                        66691 non-null  object 
 2   ended                          66691 non-null  object 
 3   issue_num                      66691 non-null  float64
 4   issue_proj                     66691 non-null  object 
 5   issue_reporter                 66691 non-null  object 
 6   issue_assignee                 35644 non-null  object 
 7   issue_contr_count              66691 non-null  float64
 8   issue_type                     66691 non-null  object 
 9   issue_priority                 66691 non-null  object 
 10  issue_created                  66691 non-null  object 
 11  issue_resolution_date          65838 non-null  object 
 12  issue_resolution               65838 non-null 

In [22]:
df_supporting[['issue_priority', 'issue_type', 'issue_resolution', 'issue_status']].describe(include='all')

,issue_priority,issue_type,issue_resolution,issue_status
count,66691,66691,65838,66691
unique,7,15,4,15
top,unknown,Ticket,Done,closed
freq,33965,45275,62034,56344


In [23]:
# Missing values - hanya kolom yang ada missing
missing_supporting = df_supporting.isnull().sum()
missing_pct = (missing_supporting / len(df_supporting) * 100).round(2)
pd.DataFrame({
    'missing_count': missing_supporting,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

,missing_count,missing_pct
wf_in_review,66614,99.88
wf_rejected,66579,99.83
wf_deployment,66556,99.80
wf_cancelled,66531,99.76
wf_pending_customer_approval,66437,99.62
wf_testing_monitoring,66401,99.57
wf_monitoring,66173,99.22
wf_to_do,65906,98.82
wf_done,65831,98.71
wf_pending_deployment,65278,97.88


In [24]:
# Rentang waktu
df_supporting['issue_created'] = pd.to_datetime(df_supporting['issue_created'], format='ISO8601', utc=True)
print('Periode:', df_supporting['issue_created'].min(), 'sampai', df_supporting['issue_created'].max())
print('Jumlah unik priority  :', df_supporting['issue_priority'].nunique(), '->', df_supporting['issue_priority'].unique())
print('Jumlah unik issue_type:', df_supporting['issue_type'].nunique(), '->', df_supporting['issue_type'].unique())

Periode: 2007-04-15 08:12:11+00:00 sampai 2023-03-14 14:00:32+00:00
Jumlah unik priority  : 7 -> ['Medium' 'High' 'unknown' 'Low' 'Blocker' 'Highest' 'Lowest']
Jumlah unik issue_type: 15 -> ['Ticket' 'Vacation' 'Story' 'Bug' 'Assistance' 'Sprint Summary'
 'Sub-task' 'Epic' 'Retrospective' 'Subtask' 'Service' 'Project' 'Task'
 'HD Service' 'Deployment']


## 6. Keterbatasan Awal Dataset

**Dataset Utama (WA_Fn-UseC_-IT-Help-Desk.csv):**
- Data simulasi, bukan data produksi nyata dari organisasi tertentu.
- Tidak ada kolom tanggal eksplisit, hanya `daysOpen` sebagai proxy durasi.
- Identitas requestor dan IT owner sudah dianonimkan.

**Dataset Pendukung (issues.csv):**
- Kolom `issue_proj`, `issue_reporter`, `issue_assignee` sudah di-mask untuk privasi.
- Kolom `wf_*` banyak missing value karena tidak semua tiket melewati setiap status workflow.
- Tidak ada informasi kategori masalah secara eksplisit, hanya `issue_type` dan `issue_priority`.

## 7. Verifikasi Struktur Raw Data

In [25]:
for root, dirs, files in os.walk('../data/raw'):
    level = root.replace('../data/raw', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f'{indent}  {f}  ({size_mb:.1f} MB)')

raw/
  supporting/
    issues_change_history.csv  (18.0 MB)
    process-flow.png  (0.0 MB)
    EXAMPLE.md  (0.0 MB)
    sample_utterances.csv  (3.2 MB)
    db.png  (0.5 MB)
    issues.csv  (19.6 MB)
    issues_snapshot.csv  (27.2 MB)
    issues_snapshot_sample.xlsx  (0.1 MB)
    FEATURES.md  (0.0 MB)
  primary/
    .gitkeep  (0.0 MB)
    WA_Fn-UseC_-IT-Help-Desk.csv  (8.0 MB)
